In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [ ]:
rec = ground_truth[0]
rec


In [ ]:
question = rec["question"]
question

In [ ]:
answer_llm = assistant.rag(question)
answer_llm

In [ ]:
assistant.total_cost()

In [ ]:
# Original answer by document id

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

In [ ]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

In [ ]:
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
# Test for 1 record

answer_record = generate_rag_answer(ground_truth[0])
answer_record

In [ ]:
assistant.reset_usage()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
results[:10]

In [ ]:
assistant.total_cost()

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
df_answers = pd.DataFrame(results)
df_answers.to_csv("data/rag-answers-new.csv", index=False)